In [26]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import dascore as dc
import os
import requests
from matplotlib.colors import CenteredNorm

def load_coords(filepath):
    """
    Loads .xycz file, fixes longitude, and calculates cumulative distance.
    Includes na_values to ensure empty spaces are recognized as NaN.
    """
    df = pd.read_csv(
        filepath, 
        sep=r'\s+', 
        header=None,
        names=['lon','lat','cha','dep'],
        na_values=['', ' ', 'NaN', 'nan']
    ).dropna(how='any')

    df['lon'] = df['lon'].apply(lambda x: x - 360 if x > 180 else x)

    # Haversine distance calculation
    R = 6371.0
    lat, lon = np.radians(df['lat'].values), np.radians(df['lon'].values)
    dlat, dlon = np.diff(lat), np.diff(lon)
    a = (np.sin(dlat/2)**2
         + np.cos(lat[:-1])
         * np.cos(lat[1:])
         * np.sin(dlon/2)**2)

    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))

    dist_km = np.insert(R * c, 0, 0).cumsum()
    df['dist_km'] = dist_km

    return df

def get_noise_rows(f_path, target_channels):
    """
    Extracts two rows: Median Absolute Amplitude of the first 5s AND the last 20s.
    """
    if not os.path.exists(f_path):
        return None, None
    try:
        patch = dc.spool(f_path)[0].detrend("time")

        t_min = patch.coords.min('time')
        t_max = patch.coords.max('time')
        t_noise_end_5s = t_min + np.timedelta64(5, 's')
        patch_5s = patch.select(time=(t_min, t_noise_end_5s))

        t_noise_start_20s = t_max - np.timedelta64(20, 's')
        patch_20s = patch.select(time=(t_noise_start_20s, t_max))

        raw_noise_5s = np.squeeze(patch_5s.abs().median(dim="time").data)
        raw_noise_20s = np.squeeze(patch_20s.abs().median(dim="time").data)

        row_5s = np.full(len(target_channels), np.nan)
        row_20s = np.full(len(target_channels), np.nan)

        for i, cha_idx in enumerate(target_channels):
            cha_idx = int(cha_idx)
            if cha_idx >= 74 and 0 <= cha_idx < len(raw_noise_5s):
                row_5s[i] = raw_noise_5s[cha_idx]
                row_20s[i] = raw_noise_20s[cha_idx]

        return row_5s, row_20s
    except Exception as e:
        print(f"Error processing {os.path.basename(f_path)}: {e}")
        return None, None

def get_magnitude(event_id, session):
    """
    Fetches earthquake magnitude from USGS API using the event ID.
    """
    url = f"https://earthquake.usgs.gov/fdsnws/event/1/query?eventid={event_id}&format=geojson"
    try:
        response = session.get(url, timeout=5)
        if response.status_code == 200:
            data = response.json()
            return data['properties']['mag']
    except Exception as e:
        print(f"Error fetching magnitude for {event_id}: {e}")
    return -1.0  # Fallback to prevent sorting failure

def plot_and_save(terra_coords, data_matrix, valid_labels, title, output_path):
    """
    Helper to generate the dual-axes plot (Bathymetry + Heatmap) without colorbar,
    using an evenly spaced visual x-axis and CenteredNorm for deviations.
    """
    avg_row = np.nanmean(data_matrix, axis=0)
    data_matrix_with_avg = [avg_row] + data_matrix
    labels_with_avg = ["Event Average"] + valid_labels

    Z = np.array(data_matrix_with_avg)

    actual_dist = terra_coords['dist_km'].values
    num_channels = len(actual_dist)

    X_idx = np.arange(num_channels)
    X_edges = np.arange(num_channels + 1) - 0.5
    Y_edges = np.arange(len(labels_with_avg) + 1)

    fig, (ax_bath, ax_data) = plt.subplots(2, 1, figsize=(14, 10),
                                           gridspec_kw={'height_ratios': [1, 6]}, 
                                           sharex=True)
    plt.subplots_adjust(hspace=0.02, bottom=0.1)

    ax_bath.plot(X_idx, -terra_coords['dep'], color='black', lw=1.5)
    ax_bath.axhline(0, color='blue', linestyle='--', lw=1)
    ax_bath.text(max(X_idx)*0.01, -1, " Sea Level", color='blue', va='bottom', fontsize=7.5)
    ax_bath.invert_yaxis()
    ax_bath.set_ylabel("Depth (m)", fontsize=10, fontweight='bold')
    ax_bath.set_title(title, fontsize=14, pad=10)
    ax_bath.grid(True, alpha=0.15, linestyle=':')

    mesh = ax_data.pcolormesh(X_edges, Y_edges, Z,
                               norm=CenteredNorm(),
                               cmap='RdBu_r', shading='flat')

    ax_data.set_yticks(np.arange(len(labels_with_avg)) + 0.5)
    ax_data.set_yticklabels(labels_with_avg, fontsize=7.5)

    yticklabels = ax_data.get_yticklabels()
    yticklabels[0].set_fontweight('bold')
    yticklabels[0].set_color('red')

    ax_data.invert_yaxis()
    ax_data.set_ylabel("Catalog Event ID", fontsize=11, fontweight='bold')

    num_ticks = 10
    tick_indices = np.linspace(0, num_channels - 1, num_ticks, dtype=int)
    tick_labels = [f"{actual_dist[idx]:.1f}" for idx in tick_indices]

    ax_data.set_xticks(tick_indices)
    ax_data.set_xticklabels(tick_labels)
    ax_data.set_xlabel("Distance along cable (km)", fontsize=11, fontweight='bold', labelpad=5)    

    print(f"Saving figure to: {output_path}")
    plt.savefig(output_path, bbox_inches='tight', dpi=300)
    plt.close(fig)


base_dir = "/Users/ed/research_code/das"
noise_game_dir = os.path.join(base_dir, "das_records/good-events-3.2-up")
coords_path = os.path.join(base_dir, 'das_coords_bathymetry/TERRA_coords.xycz')

figures_dir = os.path.join(base_dir, "das_figures")
os.makedirs(figures_dir, exist_ok=True)

good_terra_events = [
    "ak0237eejw69_TERRA.h5", "ak023aw5mbdk_TERRA.h5", "ak023eccchzy_TERRA.h5",
    "ak0237q2shdo_TERRA.h5", "ak023bebgmhd_TERRA.h5", "ak023em715sv_TERRA.h5",
    "ak02381ibekf_TERRA.h5", "ak023bhlw02w_TERRA.h5", "ak023f7eyaqg_TERRA.h5",
    "ak0238ghnzxp_TERRA.h5", "ak023bkx215x_TERRA.h5", "ak023fhgggc6_TERRA.h5",
    "ak0238qkcxek_TERRA.h5", "ak023btef8mo_TERRA.h5", "ak023fnzshe1_TERRA.h5",
    "ak0239af45c3_TERRA.h5", "ak023bzqw7a7_TERRA.h5", "ak023frilkvn_TERRA.h5",
    "ak0239lyp68s_TERRA.h5", "ak023c3206y0_TERRA.h5", "ak023g35jxin_TERRA.h5",
    "ak0239qzbmym_TERRA.h5", "ak023cgc5fmi_TERRA.h5", "ak023gbgys2j_TERRA.h5",
    "ak0239saxy95_TERRA.h5", "ak023ctiuyia_TERRA.h5", "ak023gjh7z4b_TERRA.h5",
    "ak0239vxdtm6_TERRA.h5", "ak023d3dyqv0_TERRA.h5", "ak023godcr3i_TERRA.h5",
    "ak023a0j9eo0_TERRA.h5", "ak023gqcxl3z_TERRA.h5", "ak023a7ds9th_TERRA.h5",
    "ak023dif8i7c_TERRA.h5", "ak023a91bmgs_TERRA.h5", "ak023djxyhod_TERRA.h5",
    "us6000lggw_TERRA.h5", "ak023ah9zcn5_TERRA.h5", "ak023dk7iyjo_TERRA.h5",
    "ak023asybefb_TERRA.h5", "ak023ds94esw_TERRA.h5"
]

terra_coords = load_coords(coords_path)
target_channels = terra_coords['cha'].values

req_session = requests.Session()
extracted_data = []

for fname in good_terra_events:
    event_id = fname.split('_')[0]
    row_5s, row_20s = get_noise_rows(os.path.join(noise_game_dir, fname), target_channels)

    if row_5s is not None and row_20s is not None:
        mag = get_magnitude(event_id, req_session)
        extracted_data.append((mag, event_id, row_5s, row_20s))

if not extracted_data:
    print("Error: No data found.")
else:
    extracted_data.sort(key=lambda x: x[0], reverse=True)
    matrix_5s = []
    matrix_20s = []
    valid_labels = []

    for mag, event_id, row_5s, row_20s in extracted_data:
        matrix_5s.append(row_5s)
        matrix_20s.append(row_20s)
        if mag != -1.0: 
            label = f"M {mag:.1f} - {event_id}"
        else:
            label = f"M ? - {event_id}"

        valid_labels.append(label)

    matrix_5s = np.array(matrix_5s)
    matrix_20s = np.array(matrix_20s)
    median_5s = np.nanmedian(matrix_5s, axis=0)
    norm_matrix_5s = matrix_5s - median_5s
    median_20s = np.nanmedian(matrix_20s, axis=0)
    norm_matrix_20s = matrix_20s - median_20s

    plot_and_save(terra_coords, norm_matrix_5s.tolist(), valid_labels,
                  "TERRA: Deviation from Median (First 5s)",
                  os.path.join(figures_dir, "TERRA_noise_deviation_first_5.png"))

    plot_and_save(terra_coords, norm_matrix_20s.tolist(), valid_labels,
                  "TERRA: Deviation from Median (Last 20s)",
                  os.path.join(figures_dir, "TERRA_noise_deviation_last_20.png"))

Saving figure to: /Users/ed/research_code/das/das_figures/TERRA_noise_deviation_first_5.png
Saving figure to: /Users/ed/research_code/das/das_figures/TERRA_noise_deviation_last_20.png


In [30]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import dascore as dc
import os
import requests

def load_coords(filepath):
    """
    Loads .xycz file, fixes longitude, and calculates cumulative distance.
    Includes na_values to ensure empty spaces are recognized as NaN.
    """
    df = pd.read_csv(
        filepath,
        sep=r'\s+',
        header=None,
        names=['lon','lat','cha','dep'],
        na_values=['', ' ', 'NaN', 'nan']
    ).dropna(how='any')

    df['lon'] = df['lon'].apply(lambda x: x - 360 if x > 180 else x)

    # Haversine distance calculation
    R = 6371.0
    lat, lon = np.radians(df['lat'].values), np.radians(df['lon'].values)
    dlat, dlon = np.diff(lat), np.diff(lon)
    a = (np.sin(dlat/2)**2
         + np.cos(lat[:-1])
         * np.cos(lat[1:])
         * np.sin(dlon/2)**2)

    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))

    dist_km = np.insert(R * c, 0, 0).cumsum()
    df['dist_km'] = dist_km

    return df

def get_noise_rows(f_path, target_channels):
    """
    Extracts Median Absolute Amplitude of the first 5s and the last 20s.
    """
    if not os.path.exists(f_path):
        return None, None
    try:
        patch = dc.spool(f_path)[0].detrend("time")

        t_min = patch.coords.min('time')
        t_max = patch.coords.max('time')

        t_noise_end_5s = t_min + np.timedelta64(5, 's')
        patch_5s = patch.select(time=(t_min, t_noise_end_5s))

        t_noise_start_20s = t_max - np.timedelta64(20, 's')
        patch_20s = patch.select(time=(t_noise_start_20s, t_max))

        raw_noise_5s = np.squeeze(patch_5s.abs().median(dim="time").data)
        raw_noise_20s = np.squeeze(patch_20s.abs().median(dim="time").data)

        row_5s = np.full(len(target_channels), np.nan)
        row_20s = np.full(len(target_channels), np.nan)

        for i, cha_idx in enumerate(target_channels):
            cha_idx = int(cha_idx)
            if cha_idx >= 74 and 0 <= cha_idx < len(raw_noise_5s):
                row_5s[i] = raw_noise_5s[cha_idx]
                row_20s[i] = raw_noise_20s[cha_idx]

        return row_5s, row_20s
    except Exception as e:
        print(f"Error processing {os.path.basename(f_path)}: {e}")
        return None, None

def get_magnitude(event_id, session):
    """
    Fetches earthquake magnitude from USGS API using the event ID.
    """
    url = f"https://earthquake.usgs.gov/fdsnws/event/1/query?eventid={event_id}&format=geojson"
    try:
        response = session.get(url, timeout=5)
        if response.status_code == 200:
            data = response.json()
            return data['properties']['mag']
    except Exception as e:
        print(f"Error fetching magnitude for {event_id}: {e}")
    return -1.0  # Fallback to prevent sorting failure

def plot_and_save(terra_coords, data_matrix, valid_labels, title, output_path, scale_factor=3):
    """
    Helper to generate the dual-axes plot (Bathymetry + Heatmap).
    Includes a colorbar and uses a robust median calculation for the colormap limits.
    """
    avg_row = np.nanmean(data_matrix, axis=0)
    data_matrix_with_avg = [avg_row] + data_matrix
    labels_with_avg = ["Event Average"] + valid_labels

    Z = np.array(data_matrix_with_avg)

    median_abs = np.nanmedian(np.abs(Z))
    vmax_scaled = median_abs * scale_factor

    actual_dist = terra_coords['dist_km'].values
    num_channels = len(actual_dist)

    X_idx = np.arange(num_channels)
    X_edges = np.arange(num_channels + 1) - 0.5
    Y_edges = np.arange(len(labels_with_avg) + 1)

    fig, (ax_bath, ax_data) = plt.subplots(2, 1, figsize=(14, 10),
                                           gridspec_kw={'height_ratios': [1, 6]}, 
                                           sharex=True)
    plt.subplots_adjust(hspace=0.02, bottom=0.1)

    ax_bath.plot(X_idx, -terra_coords['dep'], color='black', lw=1.5)
    ax_bath.axhline(0, color='blue', linestyle='--', lw=1)
    ax_bath.text(max(X_idx)*0.01, -1, " Sea Level", color='blue', va='bottom', fontsize=7.5)
    ax_bath.invert_yaxis()
    ax_bath.set_ylabel("Depth (m)", fontsize=10, fontweight='bold')
    ax_bath.set_title(title, fontsize=14, pad=10)
    ax_bath.grid(True, alpha=0.15, linestyle=':')

    mesh = ax_data.pcolormesh(X_edges, Y_edges, Z,
                               vmin=-vmax_scaled, vmax=vmax_scaled,
                               cmap='RdBu_r', shading='flat')

    cbar = fig.colorbar(mesh, ax=[ax_bath, ax_data], orientation='vertical', fraction=0.03, pad=0.02)
    cbar.set_label('Deviation from Median Amplitude', fontsize=11, fontweight='bold')

    ax_data.set_yticks(np.arange(len(labels_with_avg)) + 0.5)
    ax_data.set_yticklabels(labels_with_avg, fontsize=7.5)

    yticklabels = ax_data.get_yticklabels()
    yticklabels[0].set_fontweight('bold')
    yticklabels[0].set_color('red')

    ax_data.invert_yaxis()
    ax_data.set_ylabel("Catalog Event ID", fontsize=11, fontweight='bold')

    num_ticks = 10
    tick_indices = np.linspace(0, num_channels - 1, num_ticks, dtype=int)
    tick_labels = [f"{actual_dist[idx]:.1f}" for idx in tick_indices]

    ax_data.set_xticks(tick_indices)
    ax_data.set_xticklabels(tick_labels)
    ax_data.set_xlabel("Distance along cable (km)", fontsize=11, fontweight='bold', labelpad=5)    

    print(f"Saving figure to: {output_path}")
    plt.savefig(output_path, bbox_inches='tight', dpi=300)
    plt.close(fig)


base_dir = "/Users/ed/research_code/das"
noise_game_dir = os.path.join(base_dir, "das_records/good-events-3.2-up")
coords_path = os.path.join(base_dir, 'das_coords_bathymetry/TERRA_coords.xycz')

figures_dir = os.path.join(base_dir, "das_figures")
os.makedirs(figures_dir, exist_ok=True)

good_terra_events = [
    "ak0237eejw69_TERRA.h5", "ak023aw5mbdk_TERRA.h5", "ak023eccchzy_TERRA.h5",
    "ak0237q2shdo_TERRA.h5", "ak023bebgmhd_TERRA.h5", "ak023em715sv_TERRA.h5",
    "ak02381ibekf_TERRA.h5", "ak023bhlw02w_TERRA.h5", "ak023f7eyaqg_TERRA.h5",
    "ak0238ghnzxp_TERRA.h5", "ak023bkx215x_TERRA.h5", "ak023fhgggc6_TERRA.h5",
    "ak0238qkcxek_TERRA.h5", "ak023btef8mo_TERRA.h5", "ak023fnzshe1_TERRA.h5",
    "ak0239af45c3_TERRA.h5", "ak023bzqw7a7_TERRA.h5", "ak023frilkvn_TERRA.h5",
    "ak0239lyp68s_TERRA.h5", "ak023c3206y0_TERRA.h5", "ak023g35jxin_TERRA.h5",
    "ak0239qzbmym_TERRA.h5", "ak023cgc5fmi_TERRA.h5", "ak023gbgys2j_TERRA.h5",
    "ak0239saxy95_TERRA.h5", "ak023ctiuyia_TERRA.h5", "ak023gjh7z4b_TERRA.h5",
    "ak0239vxdtm6_TERRA.h5", "ak023d3dyqv0_TERRA.h5", "ak023godcr3i_TERRA.h5",
    "ak023a0j9eo0_TERRA.h5", "ak023gqcxl3z_TERRA.h5", "ak023a7ds9th_TERRA.h5",
    "ak023dif8i7c_TERRA.h5", "ak023a91bmgs_TERRA.h5", "ak023djxyhod_TERRA.h5",
    "us6000lggw_TERRA.h5", "ak023ah9zcn5_TERRA.h5", "ak023dk7iyjo_TERRA.h5",
    "ak023asybefb_TERRA.h5", "ak023ds94esw_TERRA.h5"
]

terra_coords = load_coords(coords_path)
target_channels = terra_coords['cha'].values

req_session = requests.Session()
extracted_data = []

for fname in good_terra_events:
    event_id = fname.split('_')[0]
    row_5s, row_20s = get_noise_rows(os.path.join(noise_game_dir, fname), target_channels)

    if row_5s is not None and row_20s is not None:
        mag = get_magnitude(event_id, req_session)
        extracted_data.append((mag, event_id, row_5s, row_20s))

if not extracted_data:
    print("Error: No data found.")
else:
    extracted_data.sort(key=lambda x: x[0], reverse=True)

    matrix_5s = []
    matrix_20s = []
    valid_labels = []

    for mag, event_id, row_5s, row_20s in extracted_data:
        matrix_5s.append(row_5s)
        matrix_20s.append(row_20s)

        if mag != -1.0: 
            label = f"M {mag:.1f} - {event_id}"
        else:
            label = f"M ? - {event_id}"

        valid_labels.append(label)

    matrix_5s = np.array(matrix_5s)
    matrix_20s = np.array(matrix_20s)
    median_5s = np.nanmedian(matrix_5s, axis=0)
    norm_matrix_5s = matrix_5s - median_5s
    median_20s = np.nanmedian(matrix_20s, axis=0)
    norm_matrix_20s = matrix_20s - median_20s

    plot_and_save(terra_coords, norm_matrix_5s.tolist(), valid_labels,
                  "TERRA: Deviation from Median (First 5s Noise)",
                  os.path.join(figures_dir, "TERRA_noise_deviation_first_5.png"),
                  scale_factor=10)
    plot_and_save(terra_coords, norm_matrix_20s.tolist(), valid_labels,
                  "TERRA: Deviation from Median (Last 20s Noise)",
                  os.path.join(figures_dir, "TERRA_noise_deviation_last_20.png"),
                  scale_factor=10)

Error fetching magnitude for ak0237q2shdo: HTTPSConnectionPool(host='earthquake.usgs.gov', port=443): Read timed out. (read timeout=5)
Saving figure to: /Users/ed/research_code/das/das_figures/TERRA_noise_deviation_first_5.png
Saving figure to: /Users/ed/research_code/das/das_figures/TERRA_noise_deviation_last_20.png


In [18]:
# B & W Plots
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import dascore as dc
import os
import requests
from matplotlib.colors import CenteredNorm
import matplotlib.ticker as ticker

def load_coords(filepath):
    """
    Loads .xycz file, fixes longitude, and calculates cumulative distance.
    Includes na_values to ensure empty spaces are recognized as NaN.
    """
    df = pd.read_csv(
        filepath, 
        sep=r'\s+', 
        header=None,
        names=['lon','lat','cha','dep'],
        na_values=['', ' ', 'NaN', 'nan']
    ).dropna(how='any').reset_index(drop=True)

    df['lon'] = df['lon'].apply(lambda x: x - 360 if x > 180 else x)

    # Haversine distance calculation
    R = 6371.0
    lat, lon = np.radians(df['lat'].values), np.radians(df['lon'].values)
    dlat, dlon = np.diff(lat), np.diff(lon)
    a = (np.sin(dlat/2)**2
         + np.cos(lat[:-1])
         * np.cos(lat[1:])
         * np.sin(dlon/2)**2)

    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))

    dist_km = np.insert(R * c, 0, 0).cumsum()
    df['dist_km'] = dist_km

    return df

def get_amplitude_rows(f_path, target_channels):
    """
    Extracts two rows: 
    1. Median Absolute Amplitude (MdAA) of the first 5 seconds (Noise).
    2. Maximum Absolute Amplitude (MaxAA) of the full timespan (Signal).
    """
    if not os.path.exists(f_path):
        return None, None
    try:
        patch = dc.spool(f_path)[0].detrend("time")

        t_min = patch.coords.min('time')
        t_noise_end = t_min + np.timedelta64(5, 's')
        noise_patch = patch.select(time=(t_min, t_noise_end))

        raw_noise = np.squeeze(noise_patch.abs().median(dim="time").data)
        raw_signal = np.squeeze(patch.abs().max(dim="time").data)
        noise_row = np.full(len(target_channels), np.nan)
        signal_row = np.full(len(target_channels), np.nan)

        for i, cha_idx in enumerate(target_channels):
            cha_idx = int(cha_idx)
            if cha_idx >= 74 and 0 <= cha_idx < len(raw_noise):
                noise_row[i] = raw_noise[cha_idx]
                signal_row[i] = raw_signal[cha_idx]

        return noise_row, signal_row
    except Exception as e:
        print(f"Error processing {os.path.basename(f_path)}: {e}")
        return None, None

def get_magnitude(event_id, session):
    url = f"https://earthquake.usgs.gov/fdsnws/event/1/query?eventid={event_id}&format=geojson"
    try:
        response = session.get(url, timeout=5)
        if response.status_code == 200:
            data = response.json()
            return data['properties']['mag']
    except Exception as e:
        print(f"Error fetching magnitude for {event_id}: {e}")
    return -1.0

def plot_normalized_data(sorted_coords, data_matrix, labels, title, output_path):
    num_events = len(labels)
    fig, axes = plt.subplots(nrows=num_events, ncols=1, figsize=(14, 1.0 * num_events), sharex=True)
    x_vals = sorted_coords['dep'].values

    if num_events == 1:
        axes = [axes]

    for i, ax in enumerate(axes):
        ax.plot(x_vals, data_matrix[i], color='black', lw=0.5)
        ax.axhline(0, color='red', linestyle='--', lw=0.8)
        ax.set_ylabel(labels[i], fontsize=8, rotation=0, ha='right', va='center', labelpad=15)

        ax.grid(True, alpha=0.3)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        if i < num_events - 1:
            ax.spines['bottom'].set_visible(False)
            ax.tick_params(axis='x', length=0)

    axes[-1].invert_xaxis()
    axes[-1].xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, pos: f"{abs(x):g}"))
    axes[-1].set_xlabel("Depth (m) [Channels sorted shallow to deep]", fontsize=11, fontweight='bold', labelpad=10)

    fig.suptitle(title, fontsize=16, fontweight='bold', y=0.99)
    print(f"Saving figure to: {output_path}")
    plt.savefig(output_path, bbox_inches='tight', dpi=300)
    plt.close(fig)

base_dir = "/Users/ed/research_code/das"
noise_game_dir = os.path.join(base_dir, "das_records/good-events-3.2-up")
coords_path = os.path.join(base_dir, 'das_coords_bathymetry/TERRA_coords.xycz')

figures_dir = os.path.join(base_dir, "das_figures")
os.makedirs(figures_dir, exist_ok=True)

good_terra_events = [
    "ak0237eejw69_TERRA.h5", "ak023aw5mbdk_TERRA.h5", "ak023eccchzy_TERRA.h5",
    "ak0237q2shdo_TERRA.h5", "ak023bebgmhd_TERRA.h5", "ak023em715sv_TERRA.h5",
    "ak02381ibekf_TERRA.h5", "ak023bhlw02w_TERRA.h5", "ak023f7eyaqg_TERRA.h5",
    "ak0238ghnzxp_TERRA.h5", "ak023bkx215x_TERRA.h5", "ak023fhgggc6_TERRA.h5",
    "ak0238qkcxek_TERRA.h5", "ak023btef8mo_TERRA.h5", "ak023fnzshe1_TERRA.h5",
    "ak0239af45c3_TERRA.h5", "ak023bzqw7a7_TERRA.h5", "ak023frilkvn_TERRA.h5",
    "ak0239lyp68s_TERRA.h5", "ak023c3206y0_TERRA.h5", "ak023g35jxin_TERRA.h5",
    "ak0239qzbmym_TERRA.h5", "ak023cgc5fmi_TERRA.h5", "ak023gbgys2j_TERRA.h5",
    "ak0239saxy95_TERRA.h5", "ak023ctiuyia_TERRA.h5", "ak023gjh7z4b_TERRA.h5",
    "ak0239vxdtm6_TERRA.h5", "ak023d3dyqv0_TERRA.h5", "ak023godcr3i_TERRA.h5",
    "ak023a0j9eo0_TERRA.h5", "ak023gqcxl3z_TERRA.h5", "ak023a7ds9th_TERRA.h5",
    "ak023dif8i7c_TERRA.h5", "ak023a91bmgs_TERRA.h5", "ak023djxyhod_TERRA.h5",
    "us6000lggw_TERRA.h5", "ak023ah9zcn5_TERRA.h5", "ak023dk7iyjo_TERRA.h5",
    "ak023asybefb_TERRA.h5", "ak023ds94esw_TERRA.h5"
]

terra_coords = load_coords(coords_path)
target_channels = terra_coords['cha'].values

req_session = requests.Session()

extracted_data = []

for fname in good_terra_events:
    event_id = fname.split('_')[0]
    n_row, s_row = get_amplitude_rows(os.path.join(noise_game_dir, fname), target_channels)

    if n_row is not None:
        mag = get_magnitude(event_id, req_session)
        extracted_data.append((mag, event_id, n_row, s_row))

if not extracted_data:
    print("Error: No data found.")
else:
    extracted_data.sort(key=lambda x: x[0], reverse=True)

    noise_matrix = []
    signal_matrix = []
    valid_labels = []

    for mag, event_id, n_row, s_row in extracted_data:
        noise_matrix.append(n_row)
        signal_matrix.append(s_row)
        if mag != -1.0:
            label = f"M {mag:.1f} - {event_id}"
        else:
            label = f"M ? - {event_id}"

        valid_labels.append(label)

    noise_matrix_np = np.array(noise_matrix)

    sorted_coords = terra_coords.sort_values(by='dep').reset_index(drop=True)
    sort_idx = np.argsort(terra_coords['dep'].values)
    sorted_noise_matrix = noise_matrix_np[:, sort_idx]

    cable_noise_median = np.nanmedian(sorted_noise_matrix, axis=0)
    normalized_noise_matrix = sorted_noise_matrix - cable_noise_median
    normalized_event_noise_scalars = np.nanmedian(normalized_noise_matrix, axis=1)

    updated_labels = []
    for i, label in enumerate(valid_labels):
        updated_labels.append(f"{label} (Med: {normalized_event_noise_scalars[i]:.2e})")

    plot_normalized_data(
        sorted_coords, 
        normalized_noise_matrix, 
        updated_labels,
        "TERRA: Normalized Median Amplitude Noise by Depth",
        os.path.join(figures_dir, "TERRA_normalized_noise_by_depth.png"))

Saving figure to: /Users/ed/research_code/das/das_figures/TERRA_normalized_noise_by_depth.png
